In [1]:
import pandas as pd
from CCA_utils import *

## Baseline Model Panel

In [16]:
study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)' , 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T = 5.0
vol_window = 52
freq = 'W'
gamma = 0.05  # dampening parameter — adjust freely

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)

## M1 Drift Adjustment: Merge Futures & Compute Convenience Yield

In [17]:
# Load futures
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv')
oil_futures['date'] = pd.to_datetime(oil_futures['date'], format='%d.%m.%Y')
oil_futures = oil_futures.sort_values('date')


oil_prices = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv').sort_values('date')
oil_prices['date'] = pd.to_datetime(oil_prices['date'], format='%m/%d/%y')
oil_prices = oil_prices.sort_values('date')


# Ensure your main panel is also sorted by date
cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

# 2. Perform the Directional Merge
# 'direction="nearest"' finds the closest date, whether it is before or after.
cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_prices[['date', 'Brent']], 
    on='date', 
    direction='nearest'
)

cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_futures[['date', 'Brent_12m']], 
    on='date', 
    direction='nearest'
)




# Compute convenience yield per row (using each country's own risk-free rate)
T_fut = 12/12
cca_panel_df['log_basis'] = np.log(cca_panel_df['Brent_12m'] / cca_panel_df['Brent'])
cca_panel_df['convenience_yield'] = cca_panel_df['risk_free_rate'] - (1/T_fut) * cca_panel_df['log_basis']

# Restore panel structure
cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
cca_panel_df

,country,date,index,cds_spread,fx_rate,domestic_rate,risk_free_rate,monetary_base_bn_local,external_debt_bn_usd,domestic_debt_bn_local,Brent,Brent_12m,log_basis,convenience_yield
0,Brazil,2014-01-05,0,187.25000,2.376426,0.1321,0.0363,249.50978,482.7710,953.069,107.07,103.06,-0.038171,0.074471
1,Brazil,2014-01-12,1,191.83000,2.358435,0.1321,0.0363,249.50978,482.7710,953.069,107.28,102.21,-0.048413,0.084713
2,Brazil,2014-01-19,2,194.64000,2.342524,0.1321,0.0363,249.50978,482.7710,953.069,106.19,101.89,-0.041336,0.077636
3,Brazil,2014-01-26,3,206.22000,2.398197,0.1321,0.0363,249.50978,482.7710,953.069,107.03,102.18,-0.046373,0.082673
4,Brazil,2014-02-02,4,205.43000,2.412662,0.1340,0.0352,222.94712,482.7710,955.827,105.61,101.45,-0.040187,0.075387
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9179,UAE (Abu Dhabi),2024-12-01,9179,39.78999,3.673095,0.0465,0.0463,152.58600,487.2754,26.753,71.88,69.86,-0.028505,0.074805
9180,UAE (Abu Dhabi),2024-12-08,9180,40.27000,3.672960,0.0465,0.0463,152.58600,487.2754,26.753,72.46,70.25,-0.030974,0.077274
9181,UAE (Abu Dhabi),2024-12-15,9181,41.14000,3.672960,0.0465,0.0463,152.58600,487.2754,26.753,73.80,71.22,-0.035585,0.081885
9182,UAE (Abu Dhabi),2024-12-22,9182,43.27000,3.673095,0.0465,0.0463,152.58600,487.2754,26.753,72.08,69.86,-0.031283,0.077583


## Run M1: Standard CCA + Convenience Yield Drift Adjustment

The iterative procedure (solve_CCA) runs as in M0 to recover V and $\sigma_V$.  
Then DD is recomputed with the modified drift: $r_f - y$ instead of $r_f$.

In [18]:
results = pd.DataFrame()

for country, group in cca_panel_df.groupby('country'):

    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']
    y = df['convenience_yield']

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            M_bn, dom_D_bn, fx_rate, r_d, r_f
        )
    ]

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    # Rolling sigma_y from convenience yield (same window and annualization)
    dy = df['convenience_yield'].diff()
    df['sigma_y'] = dy.rolling(window=vol_window).std() * ann_factor

    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}
    for i, row in df.iterrows():
        cca = solve_CCA_M1(row['LCL_usd'], row['sigma_lcl'], row['B_f'],
                           r_f.iloc[i], y.iloc[i], gamma, T)
        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca['sigma_total'])
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

START_DATE = '2015-01-01'
END_DATE = '2024-12-31'

results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

# ── Save ──


In [20]:
results[['date', 'country', 'cds_spread', 'risk_free_rate',
         'implied_V', 'implied_sigma_V', 'cca_converged',
         'B_f', 'LCL_usd', 'sigma_lcl',
         'convenience_yield',]].to_csv(
    '../output/results/M1_results_5YCDS_weekly_05_dampening.csv', index=False)

In [11]:
# M1 scenarios: sweep across convenience yield values
gamma = 0.4

scenarios_m1 = [
    (600, 0.02, 800, 0.025, 5, 0.0,    "y=0 (M0)"),
    (600, 0.02, 800, 0.025, 5, -0.30,  "y=-0.30 (deep contango)"),
    (600, 0.02, 800, 0.025, 5, -0.15,  "y=-0.15 (contango)"),
    (600, 0.02, 800, 0.025, 5, -0.05,  "y=-0.05 (mild contango)"),
    (600, 0.02, 800, 0.025, 5, 0.05,   "y=0.05 (mild backwardation)"),
    (600, 0.02, 800, 0.025, 5, 0.10,   "y=0.10 (backwardation)"),
    (600, 0.20, 800, 0.025, 5, 0.20,   "y=0.20 (tight market)"),
    (600, 0.20, 800, 0.025, 5, 0.30,   "y=0.30 (deep backwardation)"),
]

print(f"{'Scenario':<30} {'y':>7} {'γy':>7} {'V_M1':>8} {'σ_M1':>7} {'DD_M1':>7} {'sprd_M1':>8} {'conv':>5}")
print("-" * 90)

for LCL, sig_lcl, Bf, rf, T, y, label in scenarios_m1:
    cca_m1 = solve_CCA_M1(LCL, sig_lcl, Bf, rf, y, gamma, T)
    risk_m1 = compute_risk(cca_m1['V'], cca_m1['sigma_V'], Bf, rf, T)

    # M0 for reference
    cca_m0 = solve_CCA(LCL, sig_lcl, Bf, rf, T)
    risk_m0 = compute_risk(cca_m0['V'], cca_m0['sigma_V'], Bf, rf, T)

    print(f"{label:<30} {y:>7.2f} {cca_m1['V']:>8.1f} {cca_m1['sigma_V']:>7.4f} "
          f"{risk_m1['d2']:>7.3f} {risk_m1['credit_spread_bps']:>8.2f} "
          f"{'Y' if cca_m1['converged'] else 'N':>5}")

print(f"\n{'M0 reference':<30} {'':>7} {'':>7} {cca_m0['V']:>8.1f} {cca_m0['sigma_V']:>7.4f} "
      f"{risk_m0['d2']:>7.3f} {risk_m0['credit_spread_bps']:>8.2f}     Y")

Scenario                             y      γy     V_M1    σ_M1   DD_M1  sprd_M1  conv
------------------------------------------------------------------------------------------


TypeError: solve_CCA_M1() missing 1 required positional argument: 'T'